# Video-LLaVA Stage 1 Pretraining on Google Colab with Data Muling

This notebook provides an automated pipeline for **Stage 1 Multimodal Feature Alignment Pretraining** of **Video-LLaVA** using the original datasets with **Data Muling** to and from **Google Drive**.

### Datasets Used (OG Video-LLaVA):
- **Image Pretrain**: LLaVA-558K (`llava_image.zip`, annotations: `llava_image_.json`)
- **Video Pretrain**: Valley Video Pretrain (`valley_2.zip.001` - `012`, annotations: `valley_.json`)

### Why Data Muling?
Google Drive FUSE (`/content/drive`) is slow when reading millions of individual images/videos, resulting in GPU starvation and timeouts. **Data Muling**:
1. **Inbound Mule**: Caches compressed archives in your Google Drive (so they are never lost) and extracts them at high speed directly to Colab's ephemeral NVMe SSD (`/content/data`).
2. **Outbound Mule**: Trains at maximum local SSD speed (`/content/checkpoints`) while a background daemon continuously syncs checkpoints and adapter weights (`mm_projector.bin`) to Google Drive.
3. **Auto-Resume**: Seamlessly picks up from the latest Google Drive checkpoint if Colab disconnects.

## 1. Check GPU Environment

In [ ]:
!nvidia-smi

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = "/content/drive/MyDrive/Video-LLaVA"
os.makedirs(f"{DRIVE_ROOT}/datasets", exist_ok=True)
os.makedirs(f"{DRIVE_ROOT}/checkpoints", exist_ok=True)
print(f"Google Drive workspace ready at: {DRIVE_ROOT}")

## 3. Clone Repository & Install Dependencies

In [ ]:
# Navigate to workspace
%cd /content

# If running inside cloned repo or git clone:
# !git clone https://github.com/<YOUR_USERNAME>/grad_project.git
# %cd /content/grad_project

# Install dependencies (uses Colab's pre-installed PyTorch — do NOT pin torch versions)
!pip install -q --upgrade pip
!pip install -q transformers tokenizers sentencepiece shortuuid accelerate peft bitsandbytes einops einops-exts timm deepspeed huggingface_hub
!apt-get install -y -qq p7zip-full
!pip install -e .

## 4. Option A: Run Fast Demo / Verification (100 Samples)
Use this to test the full data muling and pretraining pipeline in under 2 minutes.

In [ ]:
!python scripts/colab_pretrain_muler.py \
    --action all \
    --demo_samples 100 \
    --drive_root /content/drive/MyDrive/Video-LLaVA \
    --local_scratch_dir /content/data \
    --local_output_dir /content/checkpoints/videollava-7b-pretrain \
    --num_train_epochs 1.0 \
    --save_steps 25

## 5. Option B: Run Full Video-LLaVA Pretraining with Full Datasets
This automatically downloads the official archives from HuggingFace to your Google Drive, mules and unzips them onto the local SSD (`/content/data`), auto-tunes batch size and precision for your GPU, and syncs checkpoints back to Google Drive.

In [ ]:
!python scripts/colab_pretrain_muler.py \
    --action all \
    --drive_root /content/drive/MyDrive/Video-LLaVA \
    --local_scratch_dir /content/data \
    --local_output_dir /content/checkpoints/videollava-7b-pretrain \
    --learning_rate 1e-3 \
    --num_train_epochs 1.0 \
    --save_steps 500 \
    --save_total_limit 2

## 6. Monitor Training with TensorBoard

In [ ]:
%load_ext tensorboard
%tensorboard --logdir /content/checkpoints/videollava-7b-pretrain

## 7. Verify Saved Checkpoints on Google Drive

In [ ]:
import os
drive_ckpt_dir = "/content/drive/MyDrive/Video-LLaVA/checkpoints/videollava-7b-pretrain"
print("Checkpoints saved to Google Drive:")
if os.path.exists(drive_ckpt_dir):
    for root, dirs, files in os.walk(drive_ckpt_dir):
        level = root.replace(drive_ckpt_dir, '').count(os.sep)
        indent = ' ' * 4 * level
        print(f"{indent}{os.path.basename(root)}/")
        subindent = ' ' * 4 * (level + 1)
        for f in files:
            print(f"{subindent}{f}")
else:
    print("No checkpoint directory found yet.")